In [1]:
pip install pandas sqlalchemy openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import re
import pandas as pd
from sqlalchemy import create_engine

# ─────────────────────────────────────────────────────────────
# CONFIG — path is resolved relative to THIS script's location
# ─────────────────────────────────────────────────────────────
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__)) \
    if "__file__" in globals() else os.getcwd()

RELATIVE_DATASETS = "E-Mobility datasets"

FOLDER = os.path.join(SCRIPT_DIR, RELATIVE_DATASETS)

DB_PATH = os.path.join(SCRIPT_DIR, "emobility.db")
DB_URL = f"sqlite:///{DB_PATH}"

IF_EXISTS = "replace"
SKIP_HIDDEN = True


# ─────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────
def clean_name(name: str) -> str:
    """Turn any string into a safe SQL identifier."""
    name = re.sub(r"\W+", "_", str(name))
    name = re.sub(r"_+", "_", name)
    name = name.strip("_").lower()
    if name and name[0].isdigit():
        name = "t_" + name
    return name or "table"


def robust_read_csv(path: str) -> pd.DataFrame:
    """Try multiple strategies until the CSV loads."""
    attempts = [
        {"encoding": "utf-8"},
        {"encoding": "utf-8-sig"},
        {"encoding": "latin-1"},
        {"encoding": "cp1252"},
        {"encoding": "utf-8", "sep": ";"},
        {"encoding": "latin-1", "sep": ";"},
        {"encoding": "utf-8", "sep": "\t"},
        {"encoding": "latin-1", "sep": "\t"},
        {"encoding": "utf-8", "sep": None, "engine": "python"},
        {"encoding": "latin-1", "sep": None, "engine": "python"},
        {"encoding": "utf-8", "on_bad_lines": "skip"},
        {"encoding": "latin-1", "on_bad_lines": "skip"},
        {"encoding": "utf-8", "sep": None, "engine": "python", "on_bad_lines": "skip"},
        {"encoding": "latin-1", "sep": None, "engine": "python", "on_bad_lines": "skip"},
    ]
    last_err = None
    for kwargs in attempts:
        try:
            return pd.read_csv(path, **kwargs)
        except Exception as e:
            last_err = e
    raise last_err


def load_file(path: str):
    """Dispatch to the right pandas reader based on extension."""
    lower = path.lower()
    if lower.endswith(".csv"):
        return robust_read_csv(path)
    if lower.endswith((".xlsx", ".xls")):
        return pd.read_excel(path)
    if lower.endswith(".json"):
        return pd.read_json(path)
    if lower.endswith(".parquet"):
        return pd.read_parquet(path)
    if lower.endswith(".txt"):
        try:
            return pd.read_csv(path, sep="\t", encoding="latin-1")
        except Exception:
            return pd.read_csv(path, encoding="latin-1")
    return None


def make_table_name(root: str, filename: str, base_folder: str) -> str:
    """Build a table name, prefixing subfolder path to avoid collisions."""
    rel_dir = os.path.relpath(root, base_folder)
    base = os.path.splitext(filename)[0]
    if rel_dir in (".", ""):
        return clean_name(base)
    return clean_name(f"{rel_dir}_{base}")


# ─────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────
def main():
    print(f"📂 Looking for datasets at: {FOLDER}")
    print(f"💾 Database will be saved at: {DB_PATH}\n")

    if not os.path.isdir(FOLDER):
        print(f"❌ Folder not found: {FOLDER}")
        print("   Fix RELATIVE_DATASETS in the CONFIG section.")
        print(f"   SCRIPT_DIR = {SCRIPT_DIR}")
        return

    engine = create_engine(DB_URL)

    total_ok = 0
    total_skip = 0
    total_fail = 0
    seen_tables = {}

    for root, dirs, files in os.walk(FOLDER):
        for filename in sorted(files):
            if SKIP_HIDDEN and (filename.startswith(".") or filename.startswith("~$")):
                total_skip += 1
                continue

            path = os.path.join(root, filename)

            try:
                df = load_file(path)
                if df is None:
                    print(f"⏭️  Skipped (unsupported): {path}")
                    total_skip += 1
                    continue

                if df.empty:
                    print(f"⏭️  Skipped (empty): {path}")
                    total_skip += 1
                    continue

                df.columns = [clean_name(c) for c in df.columns]
                df = df.dropna(axis=0, how="all").dropna(axis=1, how="all")

                table = make_table_name(root, filename, FOLDER)

                if table in seen_tables:
                    seen_tables[table] += 1
                    table = f"{table}_{seen_tables[table]}"
                else:
                    seen_tables[table] = 1

                df.to_sql(table, engine, if_exists=IF_EXISTS, index=False)
                print(f"✅ {os.path.relpath(path, FOLDER)}  →  table '{table}'  "
                      f"({len(df)} rows, {len(df.columns)} cols)")
                total_ok += 1

            except Exception as e:
                print(f"❌ Failed: {os.path.relpath(path, FOLDER)}")
                print(f"   {type(e).__name__}: {e}")
                total_fail += 1

    print("\n" + "─" * 60)
    print(f"Done.  ✅ {total_ok} loaded   ⏭️  {total_skip} skipped   ❌ {total_fail} failed")


# ─────────────────────────────────────────────────────────────
# VERIFY
# ─────────────────────────────────────────────────────────────
def show_tables():
    from sqlalchemy import inspect
    engine = create_engine(DB_URL)
    insp = inspect(engine)
    tables = insp.get_table_names()
    print(f"\n📚 Tables in database ({len(tables)}):")
    for t in tables:
        print(f"   - {t}")


if __name__ == "__main__":
    main()
    show_tables()

📂 Looking for datasets at: c:\Users\PC\OneDrive\Desktop\Data Science & Machine Learning\FREELANCE WORK\E-MOBILITY PROJECT (EAC & UNEP)\EAC-UNEP-E-Mobility-Analytics-Forecasting\Notebooks\E-Mobility datasets
💾 Database will be saved at: c:\Users\PC\OneDrive\Desktop\Data Science & Machine Learning\FREELANCE WORK\E-MOBILITY PROJECT (EAC & UNEP)\EAC-UNEP-E-Mobility-Analytics-Forecasting\Notebooks\emobility.db

❌ Folder not found: c:\Users\PC\OneDrive\Desktop\Data Science & Machine Learning\FREELANCE WORK\E-MOBILITY PROJECT (EAC & UNEP)\EAC-UNEP-E-Mobility-Analytics-Forecasting\Notebooks\E-Mobility datasets
   Fix RELATIVE_DATASETS in the CONFIG section.
   SCRIPT_DIR = c:\Users\PC\OneDrive\Desktop\Data Science & Machine Learning\FREELANCE WORK\E-MOBILITY PROJECT (EAC & UNEP)\EAC-UNEP-E-Mobility-Analytics-Forecasting\Notebooks

📚 Tables in database (0):


In [3]:
import sqlite3

conn = sqlite3.connect("emobility.db")

In [4]:
#getting database names
cursor = conn.cursor()
cursor.execute("""SELECT name
FROM sqlite_master
WHERE type = 'table';""")
print(cursor.fetchall())

[]


In [ ]:
import pandas as pd 
pd.read_sql("""SELECT name FROM sqlite_master WHERE type='table';""", conn)

,name


: 